# 📗 RAG 파이프라인 — 검색 품질을 재고 고치기

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

앞서 우리는 PDF 를 파싱해 자르고, 색인하고, 찾아, 출처가 붙은 답까지 만들었습니다. 그런데 그 과정에서 내린 결정들을 다시 떠올려 보세요. 청킹 전략은 **한 쪽에서 잰 유사도 한두 건**을 보고 골랐고, 검색이 잘 되는지도 **결과를 눈으로 훑어** 판단했습니다.

**눈으로 좋아 보이는 것과 실제로 잘 찾는 것은 다릅니다.** 이번 시간엔 그 결정을 **숫자로 재서** 확인합니다. 재야 고칠 수 있고, 고친 뒤에도 다시 재야 정말 나아졌는지 알 수 있습니다.

## ⏪ 복습 — 앞서 만든 것

- **파싱**: `pymupdf4llm` 으로 PDF 를 쪽 단위 마크다운으로 읽었습니다.
- **청킹**: 고정 · 겹침 · 문단 세 전략을 만들고, 눈대중으로 **문단 방식(400자)** 을 골랐습니다.
- **색인**: 청크를 임베딩해 ChromaDB 컬렉션 `guide_chunks` 에 담았습니다.
- **검색**: 질문과 가까운 청크를 문서명·쪽·거리와 함께 꺼내 봤습니다.

오늘은 **그때 만든 그 색인을 그대로 이어받아** 잽니다. 기준선은 새로 만들지 않습니다 — 재는 실험이니 **지금 돌아가고 있는 것**이 기준선이어야 합니다.

**오늘의 목표**

- [ ] 정답 라벨이 붙은 **평가셋**이 무엇이고 어떻게 만드는지 안다.
- [ ] 청크 순위를 **문서 순위로 접어야** 지표가 말이 되는 이유를 설명할 수 있다.
- [ ] `Hit@3` · `Precision@3` · `Recall@3` · `MRR@3` 을 **라이브러리 없이 직접 구현**하고, 작은 예시에서 손계산과 값이 맞는지 확인한다.
- [ ] 평균 성적표뿐 아니라 **질문별 점수표**를 만들어, 평균 뒤에 숨은 개별 실패를 찾아낸다.
- [ ] 실패한 질문 한 건을 펼쳐 **왜 못 찾았는지** 원인을 짚는다.
- [ ] 파이프라인의 **설정 하나만** 바꿔 여러 값으로 재고, 그 변화를 보고 **값을 고른다**.
- [ ] 개선을 이야기할 때 **함께 밝혀야 할 한계**가 무엇인지 안다.

아래 네 셀을 먼저 실행해 앞서 만든 도구들을 되살립니다. **이 노트북은 OpenAI 를 한 번도 부르지 않습니다** — 오늘 다루는 것은 답 생성이 아니라 그 앞 단계인 **검색**이기 때문입니다.

In [ ]:
# 이 노트북에서 계속 쓸 라이브러리입니다.
import pandas as pd

print("준비 완료")

In [ ]:
# [제공 코드] 임베딩 모델 준비 — 지난 시간에 쓴 한국어 문장 임베딩 모델입니다(불러오는 데 잠시 걸립니다).
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 (768차원)')

In [ ]:
# [제공 코드] 청킹 함수 — 이 강의 앞부분에서 만든 세 가지 청킹 전략입니다.
def chunk_fixed(text, size):
    """고정 크기(글자 수)로 자른다."""
    return [text[i:i + size] for i in range(0, len(text), size)]

def chunk_overlap(text, size, overlap):
    """앞 청크의 끝 일부를 다음 청크가 겹쳐 갖도록 자른다."""
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)]

def chunk_paragraph(text, size):
    """빈 줄로 나뉜 문단을 순서대로 모으되, size 를 넘기 직전에 끊는다."""
    chunks, cur = [], ''
    for para in [p.strip() for p in text.split('\n\n') if p.strip()]:
        if cur and len(cur) + len(para) > size:
            chunks.append(cur)
            cur = para
        else:
            cur = f'{cur}\n{para}' if cur else para
    if cur:
        chunks.append(cur)
    return chunks

In [ ]:
# [제공 코드] 색인·검색 도구 — 지난 시간(임베딩·벡터DB)에 배운 것을 함수로 묶어 둡니다.
import hashlib
from pathlib import Path

import chromadb

# 지난 시간에는 EphemeralClient(메모리)를 썼습니다. 오늘 문서는 수십 쪽이라 임베딩에 시간이 걸리니,
# PersistentClient 로 **디스크에 저장**합니다. 한 번 만들어 두면 커널을 새로 켜도 그대로 남아 있어
# 다시 임베딩하지 않습니다. (output/ 폴더는 실행 산출물이라 저장소에 올라가지 않습니다.)
CHROMA_DIR = Path('output' if Path('data').exists() else '../output') / 'chroma'
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

def index_fingerprint(ids, texts, metadatas):
    """색인에 담긴 내용을 한 줄로 요약한 지문. 무엇 하나라도 바뀌면 값이 달라진다.

    메타데이터까지 넣는 이유: 본문이 그대로여도 메타에 열이 하나 늘면(예: 쪽 번호) 낡은
    색인에는 그 열이 없다. 그걸 모르고 다시 쓰면 검색은 되는데 meta['article'] 에서 KeyError 가 난다.
    """
    parts = ['\n'.join(ids), '\n'.join(texts),
             '\n'.join(repr(sorted(m.items())) for m in metadatas)]
    return hashlib.sha256('\x00'.join(parts).encode()).hexdigest()[:16]

def make_index(ids, texts, metadatas, name):
    """청크를 임베딩해 컬렉션으로 만든다. 같은 내용으로 이미 만들어 뒀으면 그대로 다시 쓴다."""
    want = index_fingerprint(ids, texts, metadatas)

    got = chroma.get_or_create_collection(name, metadata={'hnsw:space': 'cosine', 'fp': want})
    if got.count() == len(ids) and (got.metadata or {}).get('fp') == want:
        print(f'{name}: 만들어 둔 색인을 그대로 씁니다 (청크 {got.count()}개)')
        return got

    # 개수나 지문이 다르면 문서가 바뀐 것이다 — 낡은 색인을 지우고 새로 만든다.
    chroma.delete_collection(name)
    col = chroma.create_collection(name, metadata={'hnsw:space': 'cosine', 'fp': want})

    emb = embed_model.encode(texts, normalize_embeddings=True)
    col.add(ids=ids, embeddings=emb.tolist(), documents=texts, metadatas=metadatas)

    print(f'{name}: 색인을 새로 만들었습니다 (청크 {col.count()}개)')
    return col

def search(col, query, k):
    """질문과 가장 가까운 청크 k개의 본문을 돌려준다."""
    qe = embed_model.encode([query], normalize_embeddings=True)
    res = col.query(query_embeddings=qe.tolist(), n_results=k)
    return res['documents'][0]

def search_docs(col, query, k):
    """상위 청크의 부모 문서 id 를 중복 없이 앞에서부터 k개 돌려준다(지표 계산용)."""
    qe = embed_model.encode([query], normalize_embeddings=True)
    res = col.query(query_embeddings=qe.tolist(), n_results=k * 5)
    seen = []
    for m in res['metadatas'][0]:
        if m['doc_id'] not in seen:
            seen.append(m['doc_id'])
        if len(seen) >= k:
            break
    return seen

## 1. 무엇을 재는가 — 평가셋

"검색이 잘 된다"는 말은 그 자체로는 아무것도 뜻하지 않습니다. 무엇을 물었을 때 무엇이 나와야 맞는 것인지가 정해져 있어야 잴 수 있습니다. 그러려면 **질문**과 **그 질문의 정답**이 미리 짝지어져 있어야 하고, 그 짝의 묶음을 **평가셋**이라고 부릅니다. 답안지 없이 채점할 수는 없습니다.

오늘 쓸 평가셋은 `data/guide_eval.csv` 이고 열이 둘뿐입니다.

| 열 | 뜻 |
|---|---|
| `query` | 사람이 실제로 던질 법한 질문 한 문장 |
| `relevant` | 그 질문의 답이 실려 있는 **문서 id** 목록. 두 개 이상이면 `\|` 로 잇는다 |

여기서 **문서 id** 는 앞서 만든 코퍼스의 `id` 열(`ai27`·`pp14` 등)이고, 그 하나가 **PDF 한 쪽**입니다.

In [ ]:
eval_set = pd.read_csv('data/guide_eval.csv')
guide = pd.read_csv('data/guide_docs.csv')

print("평가셋 크기:", eval_set.shape)
print("\n[앞 3문항]"); display(eval_set.head(3))

In [ ]:
# 정답 라벨이 몇 개씩 붙어 있는가 -- 이 분포를 알아야 뒤에서 Precision 과 Recall 을 제대로 읽는다.
label_counts = eval_set['relevant'].str.split('|').apply(len)

print("문항당 정답 문서 수")
print(label_counts.value_counts().sort_index())

two_labels = eval_set[label_counts == 2]
print(f"\n정답이 두 개인 문항: {len(two_labels)}건")
for _, row in two_labels.iterrows():
    print(f"   {row['relevant']:12s} {row['query'][:40]}...")

**이 라벨은 어디서 왔나요.** 두 단계로 만들었습니다.

1. **사람이 질문을 썼습니다.** 문서 본문의 문장을 베끼지 않고, 실무자가 던질 법한 말투로 바꿔 썼습니다. 본문을 그대로 베끼면 검색이 자기 자신을 찾는 꼴이라 점수가 부풀려집니다.
2. **정답 라벨은 실제 검색으로 확인했습니다.** 후보 질문을 색인에 던져 상위 결과를 눈으로 읽고, **그 쪽이 정말 답을 담고 있을 때만** 라벨로 채택했습니다. 소제목만 그럴듯한 쪽은 버렸습니다.

라벨이 **두 개**인 문항이 있는 이유도 여기 있습니다. 이 표준안은 작성지침 장과 표준안 전문 장이 같은 항목을 두 번 다루고, 안내서에도 한 이야기가 두 쪽에 이어진 대목이 있습니다. **둘 다 답이면 둘 다 정답으로** 달아야 라벨이 정직합니다. 하나만 달면 옳게 찾은 검색을 틀렸다고 채점하게 됩니다.

> 만드는 규칙은 `scripts/build_day16_eval.py` 에 적어 두었습니다. **평가셋을 만드는 일 자체가 실무의 큰 몫**입니다 — 모델을 고르는 것보다 오래 걸립니다.

### 잴 대상 — 앞서 만든 그 색인 그대로

재려면 **바꾸지 않은 상태**가 먼저 있어야 합니다. 앞 시간에 만든 것과 **똑같은 규칙**으로 색인을 되살립니다 — 문단 청킹 400자, 컬렉션 이름 `guide_chunks`. 내용이 같으면 다시 임베딩하지 않고 그대로 씁니다.

> **왜 굳이 그때 것을 쓸까요?** 개선 실험의 기준선은 **지금 돌아가고 있는 것**이어야 하기 때문입니다. 여기서 기준선을 다른 설정으로 새로 고르면, 나중에 "좋아졌다"고 말할 때 그것이 바꾼 설정 덕인지 **기준선을 유리하게 골라서인지** 구별할 수 없습니다.

In [ ]:
SIZE = 400

def make_chunks(chunk_fn):
    """코퍼스 전체를 잘라 (id, 본문, 메타) 세 목록으로 돌려준다."""
    ids, texts, metas = [], [], []
    for _, row in guide.iterrows():
        for i, chunk in enumerate(chunk_fn(row['본문'])):
            ids.append(f"{row['id']}-{i}")
            texts.append(chunk)
            metas.append({'doc_id': row['id'], '문서': row['문서'],
                          '발간': int(row['발간']), '쪽': int(row['쪽'])})
    return ids, texts, metas

base_ids, base_texts, base_metas = make_chunks(lambda text: chunk_paragraph(text, SIZE))
base_col = make_index(base_ids, base_texts, base_metas, 'guide_chunks')

print(f"쪽 {len(guide)}개 -> 청크 {len(base_texts)}개")

### 청크 순위를 문서 순위로 접는다

**여기서 한 번 걸려 넘어집니다.** 우리가 색인한 것은 **청크**인데, 평가셋의 정답은 **문서(쪽)** 로 적혀 있습니다. 단위가 다릅니다.

그냥 청크 순위로 재면 어떤 일이 생기는지 직접 봅시다.

In [ ]:
sample_query = eval_set.iloc[1]['query']
print(f'질문: {sample_query}')
print(f"정답 라벨: {eval_set.iloc[1]['relevant']}\n")

# 상위 청크 8개가 각각 어느 문서에서 왔는지 본다.
q_emb = embed_model.encode([sample_query], normalize_embeddings=True)
res = base_col.query(query_embeddings=q_emb.tolist(), n_results=8)

for rank, (chunk_id, meta) in enumerate(zip(res['ids'][0], res['metadatas'][0]), 1):
    print(f"{rank}위 청크 {chunk_id:9s} -> 문서 {meta['doc_id']}")

top_doc_ids = [meta['doc_id'] for meta in res['metadatas'][0]]
print(f'\n청크 8개인데 서로 다른 문서는 {len(set(top_doc_ids))}개뿐입니다.')

**같은 문서가 여러 번 나옵니다.** 한 쪽이 조각 여럿으로 갈렸으니 당연한 일입니다. 심지어 **정답 문서가 두 자리를 차지**했습니다. 그런데 이대로 "상위 3개 중 정답이 있는가"를 재면, **상위 3개가 사실은 문서 한두 개**인 셈이라 점수가 실제보다 너그러워집니다. 사용자에게 보여 줄 때도 같은 쪽을 두 번 보여 주는 셈이라 쓸모가 없습니다.

그래서 **청크 순위를 문서 순위로 접습니다** — 위에서부터 훑으며 **처음 보는 문서만** 담아 K개를 채웁니다. 앞서 만들어 둔 `search_docs` 가 하는 일이 이것입니다.

<img src="images/평가셋_문서순위.png" width="900">

In [ ]:
# search_docs 는 청크를 넉넉히 뽑아(K의 5배) 앞에서부터 처음 보는 문서만 K개 담아 돌려준다.
top_docs = search_docs(base_col, sample_query, 3)

print(f'문서 순위 상위 3: {top_docs}')
print(f"정답 라벨      : {eval_set.iloc[1]['relevant'].split('|')}")

### 🖐️ 함께 따라하기

**다른 문항**으로 같은 확인을 해 봅니다. 대상은 평가셋의 **12번째 행**(`eval_set.iloc[11]`, 정답이 두 개인 문항)입니다.

1. 그 행의 `query` 와 `relevant` 를 각각 변수에 담아 출력한다.
2. `base_col.query(...)` 로 상위 **청크 8개**를 뽑아, 각 청크가 어느 `doc_id` 에서 왔는지 출력한다.
3. 청크 8개가 **문서 몇 개**로 줄어드는지 세어 출력한다(`set` 을 쓰면 편하다).
4. `search_docs(base_col, 질문, 3)` 으로 문서 순위 3개를 뽑아, 정답 라벨 **두 개가 모두** 들어 있는지 확인해 출력한다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. 청크 순위를 그대로 두고 "상위 3개 중 정답이 있는가"를 재면, 점수가 실제보다 **너그럽게** 나오는 이유는 무엇인가요?
2. 한 질문에 정답 문서를 **두 개** 달아야 하는 상황은 어떤 경우인가요?

<details><summary>정답 보기</summary>

1. 한 문서가 청크 여러 개로 상위를 채울 수 있어서, 상위 3개가 사실은 **문서 한두 개**일 수 있다. 서로 다른 문서를 세 개 본 것처럼 채점되므로 실제보다 후하게 매겨진다.
2. **두 쪽이 정말로 같은 답을 담고 있을 때**. 한쪽만 정답으로 달면, 다른 쪽을 옳게 찾아온 검색을 틀렸다고 채점하게 된다.

</details>

## 2. 네 지표를 손으로 만든다

기준선을 재려면 지표가 필요합니다. 지표를 만들어 주는 라이브러리는 많지만, 남이 만든 함수가 낸 숫자는 **틀려도 틀린 줄 모릅니다.** 어차피 네 줄짜리 공식이니 직접 만들어 보겠습니다. 한 번 만들어 보면 그 숫자가 무엇을 세고 있는지 잊히지 않습니다.

**규약은 하나입니다 — 같은 상위 3개를 놓고 네 가지로 봅니다.** 앞 절에서 만든 문서 순위 상위 3개를 한 번 뽑아 두고, 네 지표가 각각 그것을 다른 각도에서 셉니다.

| 지표 | 답하는 질문 | 공식 |
|---|---|---|
| `Hit@3` | **찾긴 했나** | 상위 3개에 정답이 하나라도 있으면 1, 없으면 0 |
| `Precision@3` | **가져온 것 중 쓸 만한 비율** | 상위 3개 중 정답 수 ÷ 3 |
| `Recall@3` | **놓친 것 없나** | 상위 3개 중 정답 수 ÷ 전체 정답 수 |
| `MRR@3` | **몇 번째에 있었나** | 첫 정답 순위의 역수(1위면 1, 2위면 1/2, 3위면 1/3, 없으면 0) |

**Hit 과 Precision 의 차이**가 헷갈립니다. 상위 3개 중 정답이 1개일 때 — Hit 은 **1**(찾긴 했으니까), Precision 은 **1/3**(셋 중 하나만 쓸 만하니까)입니다. 같은 결과를 두 각도에서 보는 것입니다.

<img src="images/지표_네가지.png" width="900">

In [ ]:
def hit_at_k(predicted, relevant, k):
    """상위 k개 중 관련 문서가 하나라도 있으면 1, 없으면 0."""
    return 1 if any(p in relevant for p in predicted[:k]) else 0

def precision_at_k(predicted, relevant, k):
    """상위 k개 중 관련 문서의 비율(관련 수 / k)."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / k

def recall_at_k(predicted, relevant, k):
    """전체 관련 문서 중 상위 k개가 찾아낸 비율(관련 수 / 전체 관련 수)."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / len(relevant)

def mrr(predicted, relevant):
    """첫 번째 관련 문서 순위의 역수. 목록 안에 없으면 0."""
    for i, p in enumerate(predicted, 1):
        if p in relevant:
            return 1 / i
    return 0.0

print("네 지표 준비 완료")

**공식이 맞는지 손으로 확인합니다.** 가짜 검색 결과를 하나 만들어, 종이에 계산한 값과 함수가 낸 값을 나란히 놓습니다. 이 대조를 건너뛰면, 나중에 이상한 점수가 나와도 지표가 틀렸는지 검색이 나쁜지 구별할 수 없습니다.

검색 결과가 `['ai10', 'pp20', 'ai33', 'pp7']` 이고 정답이 `['ai33', 'pp7']` 두 개라고 합시다. K는 3입니다.

- **Hit@3** — 상위 3개(`ai10`·`pp20`·`ai33`)에 정답 `ai33` 이 있다 → **1**
- **Precision@3** — 상위 3개 중 정답은 `ai33` 하나 → 1 ÷ 3 = **0.333**
- **Recall@3** — 정답 2개 중 상위 3개가 찾은 것은 1개 → 1 ÷ 2 = **0.500** (`pp7` 은 4위라 놓쳤다)
- **MRR@3** — 첫 정답 `ai33` 이 **3위** → 1 ÷ 3 = **0.333**

In [ ]:
# 손으로 계산한 값과 함수가 낸 값을 나란히 놓는다.
toy_predicted = ['ai10', 'pp20', 'ai33', 'pp7']
toy_relevant = ['ai33', 'pp7']

checks = [
    ('Hit@3', hit_at_k(toy_predicted, toy_relevant, 3), 1),
    ('Precision@3', precision_at_k(toy_predicted, toy_relevant, 3), 1 / 3),
    ('Recall@3', recall_at_k(toy_predicted, toy_relevant, 3), 1 / 2),
    ('MRR@3', mrr(toy_predicted[:3], toy_relevant), 1 / 3),
]

for name, got, want in checks:
    mark = '일치' if abs(got - want) < 1e-9 else '어긋남'
    print(f'{name:12s} 코드 {got:.3f} · 손계산 {want:.3f} -> {mark}')

> **왜 K=3 인가.** 사용자에게 실제로 보여 줄 근거는 서너 개입니다. 그래서 실무에서 **K=3** 을 흔히 씁니다. `mrr` 에 넘기는 목록도 **같은 상위 3개**입니다 — 네 지표가 같은 목록을 보므로 "이 세 개를 놓고 네 가지로 읽는다"가 그대로 성립합니다.

> **P@3 을 1.0 과 견주지 마세요.** 정답이 **하나뿐**인 문항은 `Precision@3` 이 아무리 잘해도 **1/3** 이 최대입니다(셋 중 하나만 정답일 수 있으니까). 우리 평가셋은 열다섯 문항 중 열두 문항이 그렇습니다. P@3 은 **바꾸기 전과 후를 견주는** 용도로만 읽습니다.

이제 평가셋 15문항을 한 번에 돌립니다.

In [ ]:
K = 3
METRICS = ['Hit@3', 'P@3', 'R@3', 'MRR@3']

def evaluate(col):
    """평가셋 전체를 돌려 문항별 점수를 DataFrame 으로 돌려준다."""
    rows = []
    for _, row in eval_set.iterrows():
        relevant = row['relevant'].split('|')
        # 네 지표가 모두 이 한 목록을 본다.
        top_k = search_docs(col, row['query'], K)
        rows.append({
            '정답': row['relevant'],
            'Hit@3': hit_at_k(top_k, relevant, K),
            'P@3': precision_at_k(top_k, relevant, K),
            'R@3': recall_at_k(top_k, relevant, K),
            'MRR@3': mrr(top_k, relevant),
            '검색결과': ', '.join(top_k),
        })
    return pd.DataFrame(rows)

base_scores = evaluate(base_col)

print("기준선 성적표 — 문단 청킹 400자")
for name in METRICS:
    print(f'   {name:6s} {base_scores[name].mean():.3f}')

**이 네 숫자가 오늘의 기준선**입니다. 앞으로 무엇을 바꾸든 이 값과 견줍니다.

그런데 **평균만 보고 끝내면 안 됩니다.** 평균은 잘한 문항과 못한 문항을 뭉개 버립니다. 실무에서 "검색 품질 0.87" 이라는 보고를 받으면 가장 먼저 물어야 할 것은 **"어느 질문이 틀렸나"** 입니다.

In [ ]:
# 평균 뒤에 숨은 개별 점수를 펼친다.
display(base_scores)

In [ ]:
# 못 찾은 문항만 골라낸다 -- 여기서부터가 개선의 출발점이다.
base_failed = base_scores[base_scores['Hit@3'] == 0]

print(f'{len(base_scores)}문항 중 {len(base_failed)}문항이 상위 3개 안에 정답을 못 넣었습니다.')
for _, row in base_failed.iterrows():
    print(f"   정답 {row['정답']:6s} · 검색결과 {row['검색결과']}")

평균은 그럴듯한데 **두 문항은 아예 못 찾았습니다.** 이 둘이 오늘 파고들 대상입니다.

> `MRR@3` 은 상위 3개만 보므로 못 찾은 문항은 전부 **0** 이 됩니다. "4위였는지 40위였는지"는 이 값으로 구별되지 않습니다. 그건 다음 절에서 **직접 순위를 파서** 확인합니다.

### 🖐️ 함께 따라하기

만든 지표로 **성적표를 다르게 썰어** 봅니다. 새 함수를 만들 필요 없이 `base_scores` 를 씁니다.

1. `MRR@3` 이 **1.0** 인 문항이 몇 개인지 세어 출력한다(정답을 **1위로** 맞힌 문항이다).
2. `MRR@3` 이 낮은 순으로 정렬해 **아래 4문항**의 `정답`·`MRR@3`·`검색결과` 를 출력한다.
3. `P@3` 이 **0.333 보다 큰** 문항만 골라 `정답`·`P@3`·`R@3` 을 출력한다.
4. 3번 문항들만 왜 더 높은 값을 받았는지 `정답` 열을 보고 한 줄 주석으로 적는다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. 상위 3개가 `['pp1', 'pp2', 'pp3']` 이고 정답이 `['pp3']` 하나일 때, `Hit@3` · `Precision@3` · `Recall@3` · `MRR@3` 은 각각 얼마인가요?
2. 평균 `Hit@3` 이 0.87 로 그럴듯해 보이는데도 **질문별 표를 반드시 봐야 하는** 이유는 무엇인가요?

<details><summary>정답 보기</summary>

1. `Hit@3` = **1** (정답이 상위 3개 안에 있다) · `Precision@3` = **1/3 ≈ 0.333** (셋 중 하나) · `Recall@3` = **1.0** (정답 1개를 다 찾았다) · `MRR@3` = **1/3 ≈ 0.333** (첫 정답이 3위).
2. 평균은 잘한 문항과 못한 문항을 뭉갠다. 어떤 질문이 왜 틀렸는지는 **개별 점수를 펼쳐야** 보이고, 고칠 곳도 거기서 나온다. 실무에서 문제가 되는 것은 평균이 아니라 **틀린 그 질문**이다.

</details>

## 3. 왜 틀렸는지 하나만 제대로 들여다본다

두 문항을 못 찾는다는 것까지 알아냈습니다. 그런데 못 찾는다는 사실만으로는 고칠 수가 없습니다. **무엇 때문에 틀렸는지**를 알아야 어느 설정을 바꿀지 정할 수 있습니다. 실패 사례를 열 건 훑는 것보다 **한 건을 끝까지 파는 편**이 낫습니다.

둘 중 **`pp29`** 를 고릅니다.

In [ ]:
target_label = 'pp29'
target_query = eval_set[eval_set['relevant'] == target_label].iloc[0]['query']
target_row = base_scores[base_scores['정답'] == target_label].iloc[0]

print(f'질문   : {target_query}')
print(f'정답   : {target_label}')
print(f"검색결과: {target_row['검색결과']}")

이제 **왜 저것들이 올라왔는지** 봅니다. 지표는 순위만 알려 주지 이유를 알려 주지 않습니다. 이유는 **거리와 본문**에 있습니다.

In [ ]:
def show_hits(col, query, n=6):
    """상위 청크를 문서·쪽·거리·길이와 함께 펼쳐 본다."""
    emb = embed_model.encode([query], normalize_embeddings=True)
    res = col.query(query_embeddings=emb.tolist(), n_results=n)
    for rank in range(len(res['ids'][0])):
        meta = res['metadatas'][0][rank]
        length = len(res['documents'][0][rank])
        print(f"{rank + 1}위 | 거리 {res['distances'][0][rank]:.3f} | "
              f"{res['ids'][0][rank]:9s} {meta['문서']} {meta['쪽']}쪽 | {length:4d}자")

show_hits(base_col, target_query)

**정답이 5위에 있습니다** — 아주 멀리 밀린 것도 아닌데 3위 안에 못 들었습니다. 그리고 눈에 띄는 것이 하나 더 있습니다: **조각들이 하나같이 깁니다.** 400자로 자른다고 했는데 800자, 1,000자짜리가 보입니다.

이게 우연인지 확인해 봅시다.

In [ ]:
# 우리가 만든 조각들이 정말 400자 이하인가?
lengths = [len(text) for text in base_texts]
over = [n for n in lengths if n > SIZE]

print(f'전체 {len(lengths)}조각 중 {SIZE}자를 넘는 조각: {len(over)}개 ({len(over) / len(lengths) * 100:.1f}%)')
print(f'가장 긴 조각: {max(lengths)}자')
print(f'넘는 조각들의 평균 길이: {sum(over) / len(over):.0f}자')

**`chunk_paragraph(본문, 400)` 은 400자를 지키지 못합니다.** 앞 시간에 이 함수를 만들 때 "문단 하나가 `size` 보다 길면 쪼개지 않는다"고 한 줄 덧붙였는데, 그 결과가 이것입니다. 함수는 **빈 줄이 있는 자리에서만** 끊습니다. 빈 줄이 없으면 아무리 길어도 한 덩어리로 둡니다.

정답 문서 `pp29` 에 그 일이 일어났는지 봅시다.

In [ ]:
answer_row = guide[guide['id'] == target_label].iloc[0]
answer_text = answer_row['본문']
base_pieces = chunk_paragraph(answer_text, SIZE)

print(f"{answer_row['문서']} {answer_row['쪽']}쪽 · 원문 {len(answer_text)}자")
print(f'문단 청킹 {SIZE}자 -> 조각 길이 {[len(piece) for piece in base_pieces]}')

# 이 쪽은 무엇으로 이루어져 있나 -- 표가 몇 줄인지 세어 본다.
table_lines = [line for line in answer_text.splitlines() if line.strip().startswith('|')]
paragraphs = [p for p in answer_text.split('\n\n') if p.strip()]
print(f'\n전체 {len(answer_text.splitlines())}줄 중 표(|로 시작) {len(table_lines)}줄')
print(f'빈 줄로 나뉜 문단 수: {len(paragraphs)}개')

In [ ]:
# 정답의 근거가 되는 낱말이 어느 조각에 들어 있나.
for word in ['5년', '매수인']:
    at = answer_text.find(word)
    holder = [i for i, piece in enumerate(base_pieces) if word in piece]
    print(f"'{word}' 원문 {at}번째 글자 -> {holder}번 조각 안")

print()
for rank, chunk_id in enumerate(
        base_col.query(query_embeddings=embed_model.encode([target_query],
                                                           normalize_embeddings=True).tolist(),
                       n_results=len(base_ids))['ids'][0], 1):
    if chunk_id.startswith(f'{target_label}-'):
        print(f'청크 {chunk_id} -> {rank}위')

**절반은 보입니다.** 이 쪽은 본문 대부분이 **마크다운 표 한 덩어리**입니다. 표 안에는 빈 줄이 없으니 `chunk_paragraph` 가 **쪼갤 자리를 찾지 못하고**, 표 전체가 **858자짜리 조각 하나**가 되었습니다. 그리고 정답의 근거(`5년`·`매수인`)는 **그 조각 안에 분명히 들어 있습니다.**

그런데 여기서 멈추면 안 됩니다. **조각 안에 있다고 모델이 그것을 읽었다는 뜻은 아닙니다.** 앞 시간에 확인한 대로 이 임베딩 모델은 **입력의 앞부분까지만 읽고 나머지는 버립니다.** 긴 글을 건네도 앞쪽만 읽고 덮어 버리는 셈이라, 뒤에 무엇이 적혀 있든 벡터는 달라지지 않습니다.

그 한도가 이 조각에 어떻게 걸리는지 직접 재 봅시다.

In [ ]:
# 앞 시간에 본 그 한도 -- 이 모델은 입력을 앞에서부터 정해진 토큰 수까지만 읽는다.
MAX_TOKENS = embed_model.max_seq_length
tokenizer = embed_model.tokenizer

def readable_range(text):
    """이 모델이 실제로 읽는 범위가 몇 번째 글자까지인지 돌려준다(다 읽으면 None)."""
    encoded = tokenizer(text, add_special_tokens=True, return_offsets_mapping=True,
                        truncation=False, verbose=False)
    n_tokens = len(encoded['input_ids'])
    if n_tokens <= MAX_TOKENS:
        return None, n_tokens
    # MAX_TOKENS 번째 토큰이 원문에서 끝나는 위치 -> 여기까지만 벡터에 들어간다.
    return encoded['offset_mapping'][MAX_TOKENS - 1][1], n_tokens

print(f'이 모델이 읽는 한도: {MAX_TOKENS}토큰\n')
for i, piece in enumerate(base_pieces):
    cut, n_tokens = readable_range(piece)
    scope = '전체' if cut is None else f'앞 {cut}자까지'
    print(f'{i}번 조각 {len(piece):4d}자 · {n_tokens:3d}토큰 -> 읽히는 범위: {scope}')

In [ ]:
# 그래서 정답의 근거는 읽혔을까?
answer_piece = base_pieces[1]
cut, _ = readable_range(answer_piece)

for word in ['5년', '매수인']:
    at = answer_piece.find(word)
    verdict = '읽힌다' if (cut is None or at < cut) else '잘려서 읽히지 않는다'
    print(f"'{word}' 1번 조각의 {at}번째 글자 -> {verdict}")

**여기가 진짜 원인입니다.** 정답의 근거는 조각 안에 있었지만 **모델이 읽는 범위 밖**이었습니다. 묻힌 게 아니라 **아예 읽히지 않은** 것입니다.

믿기 어렵다면 직접 확인해 보세요. 858자 조각과 **그 앞부분만 잘라 낸 짧은 글**을 각각 임베딩해 견주면 됩니다.

In [ ]:
# 조각 통째와 '그 앞 200자' 를 각각 벡터로 만들어 견준다.
head_only = answer_piece[:200]
whole_vec = embed_model.encode([answer_piece], normalize_embeddings=True)[0]
head_vec = embed_model.encode([head_only], normalize_embeddings=True)[0]

print(f'{len(answer_piece)}자 조각 vs 그 앞 {len(head_only)}자')
print(f'코사인 유사도: {float(whole_vec @ head_vec):.6f}')

**유사도가 1입니다 — 두 벡터가 완전히 같습니다.** 뒤에 658자를 더 붙이든 말든 벡터가 조금도 달라지지 않았다는 뜻이고, 그건 **뒷부분이 벡터에 전혀 반영되지 않았다**는 증거입니다.

정리하면 `pp29` 가 진 이유는 이렇습니다.

1. 이 쪽은 본문이 **표 한 덩어리**라 문단 청킹이 **쪼갤 빈 줄을 못 찾았다.**
2. 그래서 **858자 조각**이 만들어졌는데, 모델은 그중 **앞부분만** 읽는다.
3. 정답의 근거는 하필 그 뒤에 있었다 — **검색이 볼 수조차 없는 글자**가 된 것이다.

그러는 사이 비슷한 표를 담은 다른 쪽들(`pp36`·`pp31`·`pp34`)이 고만고만한 거리로 앞자리를 차지했습니다.

> **여기서 얻을 습관 하나.** 실패를 봤을 때 곧바로 설정부터 바꾸지 말고, **왜 저것이 올라왔는지**를 먼저 읽으세요. "조각 안에 답이 있으니 언젠가는 찾겠지"는 **틀린 위안**입니다. 지금 진단대로라면 저 858자 덩어리를 **쪼개서, 근거를 조각의 앞쪽으로 끌어와야** 합니다. 문단 경계를 못 찾는다면 **글자 수로 세는** 방식이 답이 될 수 있습니다.

### 🖐️ 함께 따라하기

**다른 실패 문항**(`ai39`)을 같은 방법으로 진단합니다.

1. `base_failed` 에서 `pp29` 가 아닌 문항의 `정답` 과 그 질문을 출력한다.
2. `show_hits` 로 상위 6개를 펼쳐, 1위와 6위의 **거리 차이가 큰지 작은지** 눈으로 본다.
3. 정답 문서의 청크가 각각 **몇 위**인지 위와 같은 방법으로 찾아 출력한다.
4. 정답 쪽을 `chunk_paragraph(본문, SIZE)` 로 잘라 **조각 길이**를 출력하고, 이 실패도 `pp29` 처럼 **너무 큰 덩어리** 때문인지 한 줄 주석으로 적는다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. `chunk_paragraph(본문, 400)` 으로 잘랐는데 1,000자가 넘는 조각이 나왔습니다. 버그인가요?
2. "정답 문장이 조각 안에 들어 있으니 검색이 언젠가는 찾을 것이다" — 이 말이 왜 틀렸나요?

<details><summary>정답 보기</summary>

1. 버그가 아니다. 이 함수는 **빈 줄이 있는 자리에서만** 끊는다. 표처럼 빈 줄 없이 이어지는 덩어리는 쪼갤 자리가 없어 통째로 한 조각이 된다. `size` 는 "넘기 직전에 끊는다"는 기준일 뿐 **상한이 아니다.**
2. 모델이 조각을 **앞에서부터 정해진 만큼만 읽기** 때문이다. 그 범위 밖에 있는 글자는 벡터에 **전혀 반영되지 않는다** — 858자 조각과 그 앞 200자의 코사인이 **1.000000** 으로 나온 것이 그 증거다. 조각에 담는 것과 모델이 읽는 것은 **다른 일**이다.

</details>

## 4. 설정 하나만 바꿔 다시 잰다

앞 절의 진단이 고칠 곳을 지목했습니다 — **큰 덩어리를 쪼개서, 근거를 조각 앞쪽으로 끌어와야** 합니다. 앞서 만들어 둔 `chunk_overlap(text, size, overlap)` 이 그 일을 합니다. 문단 경계를 보지 않고 **글자 수로 400자씩** 끊되, 앞 조각의 끝 일부를 다음 조각이 **겹쳐** 갖습니다. 표 한 덩어리도 가차 없이 쪼개지고, 경계에 걸린 내용은 겹침 덕에 양쪽에 남습니다.

바꾸는 곳은 **여기 한 군데뿐**입니다. 두 곳을 동시에 바꾸면 나아져도 어느 쪽 덕인지 모릅니다.

그런데 겹침을 얼마로 둘까요. 여기서 흔한 실수가 "적당히 80쯤" 하고 정하는 것입니다. 우리에겐 지표가 있으니 **여러 값을 재서 고릅니다.**

> 겹침 **0** 은 곧 `chunk_fixed(본문, 400)` 과 같습니다 — 겹치지 않고 글자 수로만 끊는 것이니까요. 그래서 아래 표의 맨 윗줄은 **"문단 대신 고정 크기로만 바꿨을 때"** 를 함께 보여 줍니다.

In [ ]:
OVERLAPS = [0, 40, 80, 120, 150, 200]

sweep_rows = []
sweep_scores = {}
sweep_cols = {}
for overlap in OVERLAPS:
    ids, texts, metas = make_chunks(lambda text, o=overlap: chunk_overlap(text, SIZE, o))
    col = make_index(ids, texts, metas, f'guide_ovl{overlap}')

    scores = evaluate(col)
    sweep_scores[overlap] = scores
    sweep_cols[overlap] = col
    row = {'겹침': overlap, '청크': len(texts)}
    row.update({name: round(scores[name].mean(), 3) for name in METRICS})
    # 평균만 보면 안 된다 -- 어느 문항을 틀리는지도 값마다 다르다.
    row['못 찾은 문항'] = ', '.join(scores[scores['Hit@3'] == 0]['정답'])
    sweep_rows.append(row)

sweep = pd.DataFrame(sweep_rows).set_index('겹침')
print('기준(문단 400자):', end=' ')
print(' · '.join(f'{name} {base_scores[name].mean():.3f}' for name in METRICS))
print(f"   못 찾은 문항: {', '.join(base_failed['정답'])}\n")
display(sweep)

**곡선이 순하지 않습니다.** 0에서 80까지는 올라가는데, **120에서 절벽처럼 떨어집니다.** 그러다 200에서 다시 올라옵니다. "조금씩 올리다 보면 최고점이 나온다"는 그림이 아닙니다.

**이것이 이 절의 진짜 교훈입니다.** 겹침 값에 따른 변화는 **단조롭지도 매끈하지도 않습니다.** 그러니 "겹침은 많을수록 좋다"거나 "적당히 중간값" 같은 **감으로는 고를 수 없고, 재서 골라야** 합니다. 만약 우리가 120만 시험해 보고 "겹치기는 안 되는구나" 하고 접었다면, 80에서 나던 개선을 통째로 놓쳤을 것입니다.

**고를 값은 80 입니다.** `Hit@3` 은 40과 같지만 `MRR@3` 이 더 높습니다 — 같은 문항을 맞히되 **더 위쪽 순위로** 올려 준다는 뜻입니다.

표의 **마지막 열**도 보세요. 값마다 **틀리는 문항이 다릅니다.** 평균이 같아도 **같은 검색기가 아닙니다.**

In [ ]:
CHOSEN = 80
ovl_col = sweep_cols[CHOSEN]
ovl_scores = sweep_scores[CHOSEN]

compare = pd.DataFrame({
    '기준(문단 400)': [base_scores[name].mean() for name in METRICS],
    f'채택(겹침 {CHOSEN})': [ovl_scores[name].mean() for name in METRICS],
}, index=METRICS)
compare['차이'] = compare[f'채택(겹침 {CHOSEN})'] - compare['기준(문단 400)']

display(compare.round(3))

**네 지표가 모두 올랐습니다.** 그런데 **평균이 올랐다는 말만으로는 무엇이 나아졌는지 알 수 없습니다.** 문항 이름을 대야 합니다.

In [ ]:
# 문항별로 나란히 놓고, 상태가 바뀐 것만 이름을 댄다.
diff = pd.DataFrame({
    '정답': base_scores['정답'],
    '기준 Hit': base_scores['Hit@3'],
    '채택 Hit': ovl_scores['Hit@3'],
    '기준 MRR@3': base_scores['MRR@3'].round(3),
    '채택 MRR@3': ovl_scores['MRR@3'].round(3),
})

print('[Hit 이 뒤집힌 문항]')
display(diff[diff['기준 Hit'] != diff['채택 Hit']])

print('[고친 뒤에도 여전히 못 찾는 문항]')
display(diff[(diff['기준 Hit'] == 0) & (diff['채택 Hit'] == 0)])

**뒤집힌 것은 `pp29` 하나**입니다 — §3 에서 파고들었던 바로 그 문항입니다. 진단이 처방으로 이어진 것이니, **정말 진단한 이유 때문에 고쳐졌는지** 확인해야 합니다. 짐작으로 넘어가면 안 됩니다.

In [ ]:
# pp29 를 겹침 방식으로 다시 잘라 본다 -- 858자 덩어리가 어떻게 됐나.
ovl_pieces = chunk_overlap(answer_text, SIZE, CHOSEN)

print(f'기준(문단 400)   조각 길이 {[len(p) for p in base_pieces]}')
print(f'채택(겹침 {CHOSEN})   조각 길이 {[len(p) for p in ovl_pieces]}')

# 정답 근거가 이제는 '읽히는 범위' 안으로 들어왔는가 -- 이것이 진단의 답이다.
for i, piece in enumerate(ovl_pieces):
    cut, n_tokens = readable_range(piece)
    found = []
    for word in ['5년', '매수인']:
        at = piece.find(word)
        if at >= 0:
            verdict = '읽힘' if (cut is None or at < cut) else '잘림'
            found.append(f'{word}@{at}자 {verdict}')
    scope = '전체' if cut is None else f'앞 {cut}자'
    print(f"{i}번 조각 {len(piece):3d}자 · 읽히는 범위 {scope:8s} · {' / '.join(found)}")

In [ ]:
print('[기준] pp29 를 찾을 때 올라온 조각들')
show_hits(base_col, target_query, n=5)
print('\n[채택] pp29 를 찾을 때 올라온 조각들')
show_hits(ovl_col, target_query, n=5)

**진단이 맞았습니다.** 858자 덩어리가 400자짜리 조각들로 쪼개지면서, 잘려서 못 읽히던 두 근거가 **서로 다른 조각의 앞쪽으로 옮겨 가 각각 읽히게** 되었습니다. `5년` 은 한 조각에서, `매수인` 은 다른 조각에서 읽힙니다.

그 결과가 검색 결과에 그대로 나타납니다 — **`pp29` 의 조각 두 개가 나란히** 상위로 올라왔습니다. 각각 다른 근거를 읽었기 때문입니다. 정답 조각의 거리는 0.365에서 0.340으로 줄었습니다.

**그리고 `ai39` 는 여전히 못 찾습니다.** 따라하기에서 봤듯 그 실패는 조각 크기 문제가 아니라 **질문과 문서가 같은 것을 다른 이름으로 부르는** 문제였습니다. 조각을 쪼개는 것으로는 낫지 않습니다. **다 맞았다고 끝내는 그림은 실무를 속입니다.**

### 한 문항이 고쳐졌다고 문제가 사라진 건 아니다

여기서 한 가지를 더 확인해야 합니다. 우리는 `pp29` 를 고쳤지만, **잘려 나가는 문제 자체를 없앤 것은 아닙니다.** 코퍼스 전체에서 몇 조각이나 한도를 넘는지 세어 봅시다.

In [ ]:
def over_limit(chunks):
    """모델이 읽는 한도를 넘는 조각이 몇 개인지 세어 (개수, 비율, 최대 토큰) 로 돌려준다."""
    counts = [readable_range(chunk)[1] for chunk in chunks]
    over = [n for n in counts if n > MAX_TOKENS]
    return len(over), len(over) / len(counts) * 100, max(counts)

ovl_ids, ovl_texts, ovl_metas = make_chunks(
    lambda text: chunk_overlap(text, SIZE, CHOSEN))

for name, chunks in [('기준(문단 400)', base_texts), (f'채택(겹침 {CHOSEN})', ovl_texts)]:
    n_over, ratio, biggest = over_limit(chunks)
    print(f'{name:14s} 조각 {len(chunks):3d}개 중 {n_over:3d}개({ratio:.0f}%)가 {MAX_TOKENS}토큰 초과 · 가장 긴 조각 {biggest}토큰')

**고친 뒤에도 대부분의 조각이 여전히 한도를 넘습니다.** 달라진 것은 **가장 긴 조각이 훨씬 짧아졌다**는 점입니다. 그래서 잘려 나가는 양이 줄었고, 그 덕에 `pp29` 의 근거가 읽히는 범위 안으로 들어온 것입니다.

**이 겹침 설정은 문제를 없앤 것이 아니라 줄인 것입니다.** 이 사실을 알고 있어야 다음에 무엇을 더 할지 판단할 수 있습니다 — 조각을 더 작게 만들지, 아니면 다른 방법을 쓸지는 **또 재서** 정할 일입니다.

### 정직하게 남길 것

이 절의 결과를 **"겹쳐 자르면 검색이 좋아진다"** 로 외우면 안 됩니다. 실제로 확인한 것은 그보다 훨씬 좁고, 함께 밝혀야 할 것이 넷 있습니다.

**① 평가셋이 15문항이라 한 문항이 결과를 크게 좌우합니다.** 한 문항이 뒤집히면 Hit@3 이 **0.067** 움직입니다. 위에서 본 개선폭 대부분은 사실 **한 문항의 이야기**입니다. 아래에서 직접 확인합니다.

**② 크기를 바꾸면 결론도 바뀝니다.** 크기 400 에서는 겹치기가 도움이 됐지만, 크기를 350 으로 잡으면 **오히려 나빠집니다.** 이것도 직접 재 봅니다.

**③ 기준을 어디에 두느냐에 따라 "효과 없음"과 "효과 있음"이 갈립니다.** 이건 남의 이야기가 아닙니다 — 이 실습을 준비하면서 실제로 겪은 일입니다. 같은 겹침 설정을 **다른 자료**에 걸었을 때, 어떤 기준에서 출발하면 지표가 **하나도 움직이지 않아** "이 설정은 저 자료엔 소용없다"는 결론이 나왔습니다. 그런데 **출발점을 다른 설정으로 옮기자 같은 겹침 값이 뚜렷하게 효과를 냈습니다.** 바꾼 설정은 그대로였고 **출발점만 달랐습니다.** 그래서 개선을 이야기할 때는 "무엇을 바꿨나"만큼 **"무엇에서 출발했나"** 를 반드시 함께 밝혀야 합니다.

**④ 개선 뒤에도 못 찾는 문항이 남았습니다.** 위에서 이름까지 확인했습니다.

In [ ]:
# 한계 ① -- 문항 하나가 지표를 얼마나 움직이나.
one_item = 1 / len(eval_set)
hit_delta = ovl_scores['Hit@3'].mean() - base_scores['Hit@3'].mean()

print(f'문항 하나의 무게      : {one_item:.3f}')
print(f'이번 Hit@3 개선폭     : {hit_delta:.3f}')
print(f'즉 이 개선은 문항 {round(hit_delta / one_item)}개가 뒤집힌 결과입니다.')

In [ ]:
# 한계 ② -- 같은 겹침 설정을 다른 청크 크기(350자)에서 쓰면 어떻게 되나.
SMALL = 350
small_plain = make_index(*make_chunks(lambda text: chunk_fixed(text, SMALL)), 'guide_fixed350')
small_ovl = make_index(*make_chunks(lambda text: chunk_overlap(text, SMALL, 70)), 'guide_s350ovl')

small_table = pd.DataFrame({
    '크기 350 · 겹침 0': [evaluate(small_plain)[name].mean() for name in METRICS],
    '크기 350 · 겹침 70': [evaluate(small_ovl)[name].mean() for name in METRICS],
}, index=METRICS)
small_table['차이'] = small_table['크기 350 · 겹침 70'] - small_table['크기 350 · 겹침 0']
display(small_table.round(3))

**크기 350에서는 겹치기가 지표를 끌어내립니다.** 같은 겹침 설정, 같은 평가셋, 같은 모델인데 **크기 하나가 다를 뿐인데 방향이 반대**입니다.

그러니 오늘 얻은 결론을 정확히 쓰면 이렇습니다.

> **이 코퍼스 · 이 평가셋 15문항 · 크기 400 · K=3** 에서 겹침 80 이 기준보다 나았다.

"겹치기가 정답"이 아닙니다. **어디서나 통하는 설정값은 없고, 자기 데이터로 재 봐야 압니다.** 오늘 배운 것은 겹침 청킹이 아니라 **재고 → 원인 읽고 → 한 곳 고치고 → 다시 재는 절차**입니다. 그 절차는 어느 데이터에서도 씁니다.

### 그럼 앞 시간의 선택은 틀렸던 걸까

아닙니다. 앞 시간에 고른 **문단 400자**는 이렇게 재 봐도 나쁘지 않은 설정이었습니다. 열다섯 문항 중 열셋을 맞혔습니다.

다만 두 가지가 남습니다.

- **나쁘지 않았다는 사실 자체를 재고 나서야 알았습니다.** 재기 전까지 우리는 그 설정이 좋은지 나쁜지 말할 근거가 없었습니다.
- **겹침을 조금 더하면 더 낫다는 것도 재기 전엔 알 수 없었습니다.** 오히려 앞 시간엔 겹침 방식을 "값을 못 한다"고 접었습니다. 그때 근거는 **쪽 하나에서 잰 유사도 한두 건**이었습니다.

**감으로 고른 것이 맞을 수도 있습니다. 다만 맞았는지는 재야 압니다.**

### 🖐️ 함께 따라하기

**겹침 150 을 골랐다면 어땠을까요.** 표에서 150은 기준보다 지표가 낮았습니다. 그런데 **평균이 낮다**는 말과 **무엇이 망가졌다**는 말은 다릅니다. 문항 단위로 확인해 봅니다.

1. `sweep_scores[150]` 에서 `Hit@3` 이 0인 문항의 `정답` 목록을 출력한다.
2. 기준(`base_failed`)의 실패 목록과 비교해, **기준에서는 맞혔는데 150에서 깨진** 문항을 출력한다.
3. 반대로 **기준에서는 틀렸는데 150에서 고쳐진** 문항도 출력한다.
4. 2번과 3번의 개수를 한 줄로 나란히 출력하고, 무엇이 더 많은지 한 줄 주석으로 적는다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

1. 겹침 값을 여러 개 재 보지 않고 **120 하나만** 시험했다면 어떤 결론을 내렸을까요? 그게 왜 위험한가요?
2. 동료가 "겹쳐 자르면 검색이 좋아진다더라, 우리 프로젝트에도 바로 넣자"고 합니다. 오늘 실습을 근거로 어떻게 답하겠습니까?

<details><summary>정답 보기</summary>

1. 120은 기준보다 **나빴으니** "겹치기는 이 데이터에 안 맞는다"고 접었을 것이다. 그런데 80에서는 분명히 나아졌다. 겹침 값에 따른 변화가 **단조롭지 않기** 때문에, 한 값만 보고 겹치기 자체를 판단하면 있는 개선을 통째로 놓친다.
2. "우리 데이터로 먼저 재 보자" 고 답한다. 오늘 실습에서도 **청크 크기를 350으로 바꾸자 같은 겹침 설정이 지표를 끌어내렸고**, 겹침 값에 따라 좋아졌다 나빠졌다 했다. 게다가 **어디서 출발하느냐에 따라 효과가 있기도 없기도** 했다. 평가셋을 만들어 기준선을 재고, 한 곳만 바꿔 다시 재는 절차부터 세우는 것이 순서다.

</details>

## 이번 강의 정리

| 단계 | 한 일 | 핵심 |
|---|---|---|
| 평가셋 | 질문 + 정답 문서 라벨 | 사람이 쓴 질문 · 라벨은 **실제 검색으로 확인** · 답이 둘이면 둘 다 |
| 단위 맞추기 | 청크 순위 → 문서 순위 | 한 문서가 상위를 독차지하는 것을 접는다(`search_docs`) |
| 기준선 | 앞 시간에 만든 색인 그대로 | 기준선은 **지금 돌아가고 있는 것**이어야 한다 |
| 지표 | Hit · Precision · Recall · MRR **@3** | **같은 상위 3개를 놓고 네 가지로 본다** |
| 읽는 법 | 평균 + **질문별 표** | 평균은 실패를 뭉갠다 — 먼저 물을 것은 "어느 질문이 틀렸나" |
| 진단 | 실패 한 건을 끝까지 | 문단 청킹이 표를 못 쪼개 **858자 덩어리**가 됐고, 근거는 **읽히는 범위 밖**이라 벡터에 들어가지도 않았다 |
| 개선 | 설정 **하나만**, 여러 값으로 | 겹침 값에 따른 변화가 **단조롭지도 매끈하지도 않다** — 감이 아니라 재서 고른다 |
| 정직 | 한 문항 = 0.067 · 크기 350 에선 반대 · **출발점이 결론을 바꾼다** · 남은 실패 | 배운 것은 특정 설정값이 아니라 **절차** |

가장 중요한 습관 하나만 남긴다면: **바꾸기 전에 먼저 재는 것**입니다. 기준선 없이 고치면 나아졌는지 나빠졌는지조차 말할 수 없고, 나아졌더라도 **무엇에서 나아진 것인지** 말할 수 없습니다.

## ⏭️ 예고 — 다음 단원: 데이터베이스와 SQL

오늘까지 다룬 문서는 파일에서 읽어 메모리에 올린 것이었습니다. 다음 단원에서는 자료를 **데이터베이스에 담고 질의어로 꺼내는** 법을 배웁니다.